# 2.2 — 関数・エラー・テスト

2.1で作ったレコード処理を、名前を付けて再利用でき、入力と結果を検証できる関数へ分けます。

## 導入

このNotebookでは、本文の考え方を実際のコードで確かめます。

## このレッスンの到達目標

- 引数と有用な戻り値を持つ関数を定義し、呼び出せる。
- ローカルスコープ、既定値、キーワード引数、docstring、型ヒントを関数契約として説明できる。
- 検証・計算・状態変更・表示の責任を分けられる。
- 構文・実行時・論理エラーを区別し、狭い範囲で例外を処理できる。
- 戻り値、発生する例外、呼び出し前後の状態をテストできる。

> **学習経路:** 必須：2.2.1～2.2.6　｜　統合練習：2.2.7


## 2.2.1 入力と出力が明確な関数を定義する

`def`で関数を定義します。定義しただけでは本体は実行されず、関数を呼び出したときに、インデントされた本体が実行されます。定義は呼び出しより先に実行されている必要があります。

In [ ]:
def completion_rate(completed, registered):
    return completed / registered * 100

rate = completion_rate(32, 40)
print(f"修了率: {rate:.1f}%")


### 仮引数は入口、戻り値は出口

定義に書く`completed`と`registered`は仮引数、呼び出しで渡す`32`と`40`は実引数です。`return`は計算結果を呼び出し元へ返し、その時点で関数を終了します。`print()`は画面に表示するだけなので、後の計算に使う値の代わりにはなりません。

In [ ]:
def add_with_print(a, b):
    print(a + b)

def add_with_return(a, b):
    return a + b

printed_result = add_with_print(2, 3)
returned_result = add_with_return(2, 3)
print("print版の戻り値:", printed_result)
print("return版を再計算:", returned_result * 2)


## 2.2.2 スコープを管理し、呼び出し規約を伝える

関数内で代入した変数はローカル変数で、通常は関数外から参照できません。必要な値は引数で受け取り、結果は`return`で返すと、外部状態への依存が減り、同じ入力で同じ結果を確認しやすくなります。

In [ ]:
def difference(planned, actual):
    gap = planned - actual
    return gap

print(difference(40, 34))
# print(gap)  # NameError: gapは関数の外には存在しない


### 既定値とキーワード引数は呼び出しの意味を明確にする

省略可能な仮引数には既定値を付けられます。既定値のない仮引数を先に書きます。キーワード引数を使うと、同じ型の値が並ぶ呼び出しでも意味を読み取りやすくなります。

In [ ]:
def format_centre(name, completed, registered, decimals=1):
    rate = completed / registered * 100
    return f"{name}: {rate:.{decimals}f}%"

print(format_centre("North", completed=32, registered=40))
print(format_centre("South", completed=24, registered=35, decimals=2))


### docstringと型ヒントは契約を伝える

関数直下のdocstringには目的、入力、戻り値、無効な入力の扱いを書きます。型ヒントは読み手と開発ツールへの情報であり、実行時に型を自動強制する仕組みではありません。

In [ ]:
def safe_rate(completed: int, registered: int) -> float | None:
    """修了率を返す。登録者数が0以下ならNoneを返す。"""
    if registered <= 0:
        return None
    return completed / registered * 100

print(safe_rate(32, 40))
print(safe_rate(0, 0))


## 2.2.3 検証・処理・表示の責任を分ける

レコードの検証、率の計算、表示を一つの長い処理へ混ぜず、小さな関数へ分けます。必須キーの欠落と、値の範囲違反も区別します。

In [ ]:
REQUIRED_FIELDS = {"name", "registered", "completed"}

def validate_centre(centre):
    missing = REQUIRED_FIELDS - centre.keys()
    if missing:
        raise KeyError(f"必須項目がありません: {sorted(missing)}")
    if centre["registered"] < 0 or centre["completed"] < 0:
        raise ValueError("人数を負数にはできません")
    if centre["completed"] > centre["registered"]:
        raise ValueError("修了者数が登録者数を超えています")

def centre_rate(centre):
    validate_centre(centre)
    if centre["registered"] == 0:
        return None
    return centre["completed"] / centre["registered"] * 100

centre = {"name": "North", "registered": 40, "completed": 32}
print(centre_rate(centre))


## 2.2.4 エラーを読み、予測できる例外だけを捕捉する

文法エラーは実行前に構文を解釈できない状態、実行時エラーは実行中に例外が発生した状態、論理エラーは実行できても結果が誤っている状態です。トレースバックは最後の行から例外名とメッセージを確認し、その上の自分のコード行へ戻ります。

In [ ]:
def broken_rate(completed, registered):
    return completed / registerd * 100  # 名前の綴りが違う

try:
    broken_rate(32, 40)
except NameError as error:
    print(type(error).__name__)
    print(error)


### 予想できる例外だけを狭く捕捉する

`try`には失敗し得る最小範囲を置き、`except`では対処できる具体的な例外を指定します。`else`は例外がなかった場合、`finally`は成否にかかわらず必要な後始末に使います。原因を隠す`except Exception: pass`は避けます。

In [ ]:
raw = "40"
try:
    registered = int(raw)
except ValueError:
    print("整数として読めません")
else:
    print("登録者数:", registered)
finally:
    print("入力確認を終了しました")


## 2.2.5 正常・境界・異常ケースをテストする

正常値だけでは境界のバグを見つけられません。`assert`で期待値を明記し、通常の値、0などの境界、無効な値を確認します。浮動小数点数は完全一致ではなく許容誤差で比較します。`assert`は学習時の検査には便利ですが、利用者入力の検証の代わりにはしません。

In [ ]:
assert abs(safe_rate(32, 40) - 80.0) < 0.0001
assert safe_rate(0, 0) is None
assert safe_rate(1, -1) is None

try:
    validate_centre({"name": "Bad", "registered": 5, "completed": 7})
except ValueError:
    pass
else:
    raise AssertionError("ValueErrorを期待しました")

print("すべてのテストに合格しました")


## 2.2.6 検索・更新関数の契約と状態変化をテストする

2.1の備品台帳を使い、`find_asset(assets, asset_id)`、`add_asset(...)`、`mark_available(...)`、`remove_asset(...)`へ分けてください。検索の該当なしは`None`、空欄・重複追加は`ValueError`、存在しない更新・削除は`KeyError`とします。正常値と異常値に加え、件数と順序の変更前後も`assert`で確認します。


In [ ]:
def find_book(books, book_id):
    for book in books:
        if book["id"] == book_id:
            return book
    return None

def mark_as_read(books, book_id):
    book = find_book(books, book_id)
    if book is None:
        raise KeyError(book_id)
    book["read"] = True
    return book

books = [{"id": "B001", "title": "Python Basics", "read": False}]
changed = mark_as_read(books, "B001")
print(changed)
print(books)


### 値の規則違反と、更新対象の欠落を分ける

空のIDや重複IDは、新しく渡された値が追加規則に反するため`ValueError`です。一方、更新や削除を依頼されたIDが存在しない場合は、対象キーがないため`KeyError`とします。例外名を分けると、確認プログラムは失敗理由まで検査できます。

In [ ]:
def add_book(books, book_id, title):
    clean_id = book_id.strip()
    clean_title = title.strip()
    if not clean_id or not clean_title:
        raise ValueError("id and title are required")
    if find_book(books, clean_id) is not None:
        raise ValueError(f"duplicate id: {clean_id}")
    book = {"id": clean_id, "title": clean_title, "read": False}
    books.append(book)
    return book


### 戻り値だけでなく、呼び出し前後の状態をテストする

リストを変更する関数では、返された辞書だけを確認しても不十分です。件数が一つ増えたか、返された辞書が実際にリストへ格納されたか、既存レコードの順序が保たれたかを確認します。計算だけを行う関数なら、逆に入力を変更していないことを確認します。

In [ ]:
before_count = len(books)
added = add_book(books, " B002 ", " Working with Data ")
assert len(books) == before_count + 1
assert added is books[-1]
assert added == {"id": "B002", "title": "Working with Data", "read": False}

try:
    add_book(books, "B002", "Duplicate")
except ValueError:
    pass
else:
    raise AssertionError("ValueErrorを期待しました")

print("STATE TESTS PASSED")


### 確認プログラムは関数契約を利用する別のプログラム

学習者がこの段階で確認プログラムの内部を作れる必要はありません。確認プログラムは`library_manager.py`を読み込み、決められた関数を通常値・境界値・異常値で呼び出します。学習者はファイル名、関数名、引数、戻り値、例外を契約どおりに保ち、`NG`なら自分のプログラムだけを修正します。

## 2.2.7 統合練習：テスト済み関数を接続する

2.1の3センターのリストを使い、`validate_centre()`、`centre_rate()`、`summarise_centres()`へ処理を分けてください。最後の関数は、修了率75%未満のセンター名、地区の集合、全体の登録者数と修了者数を辞書で返します。正常な3件、登録者数0、必須キー欠落、修了者数超過をテストします。

In [ ]:
# ここに応用練習の解答を書きます。


## まとめ

- 各処理責任に安定した名前と契約を与えました。
- 予測できる例外だけを、失敗する処理の近くで扱いました。
- 正常・境界・異常・状態変更を個別に検証しました。

## 次のレッスンへ

2.3では、関数とエラーの契約を使い、CSVから永続的なレコードを読み、別の検証可能な結果として保存します。

**学習時間の目安:** 約3時間
